# 02 — Data preparation evidence

Preparation and validation evidence for the canonical geometry, governed CLC derivatives, and national panel. National preparation logic lives in reusable `src/` modules and scripts; this notebook does not modify raw or processed data.

In [ ]:
from pathlib import Path
import json
import sys
import pyogrio

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SPATIAL
from src.geospatial_utils import GRID_PATH
from src.source_registry import CLC_PREPARED_PORTUGAL_LAYERS
print(SPATIAL)

## Canonical grid geometry

In [ ]:
grid_info = pyogrio.read_info(GRID_PATH, layer='canonical_mainland_grid_1km')
assert grid_info['features'] == 89_112
assert grid_info['crs'] == 'EPSG:3763'
print({'path': GRID_PATH.relative_to(PROJECT_ROOT).as_posix(), 'layer': 'canonical_mainland_grid_1km', 'features': grid_info['features'], 'crs': grid_info['crs']})

## Governed Portugal CLC layers

In [ ]:
for year, record in CLC_PREPARED_PORTUGAL_LAYERS.items():
    path = PROJECT_ROOT / record.prepared_path
    facts = record.validation_facts
    info = pyogrio.read_info(path, layer=facts.layer_name)
    assert info['features'] == facts.feature_count
    assert info['crs'] == record.crs
    assert facts.class_code_field in info['fields']
    print(year, record.prepared_path, info['features'], info['crs'], 'registered validation: passed')

## National-panel validation evidence

In [ ]:
metrics_path = PROJECT_ROOT / 'data/processed/national_panel_2015_2024_validation.json'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print({'rows': metrics['actual_row_count'], 'cells': metrics['grid_cell_count'], 'duplicate_keys': metrics['duplicate_analytical_key_count']})